In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import (
    SimpleImputer,
    KNNImputer
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

# =====================================================
# CREATE DATASET
# =====================================================

df = pd.DataFrame({

    'Age':[25,40,np.nan,35,29,50,31,np.nan,27,45],

    'Income':[
        30000,90000,55000,np.nan,
        48000,120000,62000,
        58000,np.nan,75000
    ],

    'LoanAmount':[
        10000,50000,20000,30000,
        np.nan,70000,25000,
        22000,18000,40000
    ],

    'CreditScore':[
        650,780,720,690,
        710,800,np.nan,
        705,680,790
    ],

    'EmploymentType':[
        'Private',
        'Government',
        'Private',
        'Business',
        'Private',
        'Government',
        'Business',
        'Private',
        'Business',
        'Government'
    ],

    'Education':[
        'Bachelor',
        'Master',
        'Bachelor',
        'PhD',
        'Bachelor',
        'PhD',
        'Master',
        np.nan,
        'Bachelor',
        'Master'
    ],

    'CityCode':[
        'NY-1001',
        'LA-2002',
        'TX-3003',
        'NY-4004',
        'CA-5005',
        'TX-6006',
        'LA-7007',
        'NY-8008',
        'CA-9009',
        'TX-1010'
    ],

    'MaritalStatus':[
        'Single',
        'Married',
        'Single',
        'Married',
        'Single',
        'Married',
        'Single',
        'Divorced',
        'Married',
        'Single'
    ],

    'Default':[0,0,1,0,1,0,1,1,0,0]
})

df

,Age,Income,LoanAmount,CreditScore,EmploymentType,Education,CityCode,MaritalStatus,Default
0,25.0,30000.0,10000.0,650.0,Private,Bachelor,NY-1001,Single,0
1,40.0,90000.0,50000.0,780.0,Government,Master,LA-2002,Married,0
2,NaN,55000.0,20000.0,720.0,Private,Bachelor,TX-3003,Single,1
3,35.0,NaN,30000.0,690.0,Business,PhD,NY-4004,Married,0
4,29.0,48000.0,NaN,710.0,Private,Bachelor,CA-5005,Single,1
5,50.0,120000.0,70000.0,800.0,Government,PhD,TX-6006,Married,0
6,31.0,62000.0,25000.0,NaN,Business,Master,LA-7007,Single,1
7,NaN,58000.0,22000.0,705.0,Private,NaN,NY-8008,Divorced,1
8,27.0,NaN,18000.0,680.0,Business,Bachelor,CA-9009,Married,0
9,45.0,75000.0,40000.0,790.0,Government,Master,TX-1010,Single,0


In [9]:
# extract city category

df['city_cat'] = df['CityCode'].apply(
    lambda s: s.split('-')[0]
)

# extract city number

df['city_num'] = df['CityCode'].apply(
    lambda s: s.split('-')[-1]
)

df['city_num'] = pd.to_numeric(
    df['city_num'],
    errors='coerce'
)

# drop original column

df.drop(columns=['CityCode'], inplace=True)

# =====================================================
# X AND y
# =====================================================

X = df.drop(columns=['Default'])

y = df['Default']

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42
)

# =====================================================
# NUMERICAL FEATURES
# =====================================================

num_features = [
    'Age',
    'Income',
    'LoanAmount',
    'CreditScore',
    'city_num'
]

# =====================================================
# WHY KNN HERE?
# =====================================================

# Because:
# Income, CreditScore, LoanAmount
# are related to each other.
#
# Similar customers tend to have
# similar financial patterns.
#
# So KNN can estimate missing values
# better than simple mean sometimes.

# =====================================================
# KNN NUMERICAL PIPELINE
# =====================================================

num_pipeline = Pipeline([

    ('knn',
     KNNImputer(
         n_neighbors=3,
         weights='distance'
     )),

    ('scaler',
     StandardScaler())

])

# =====================================================
# ORDINAL FEATURE
# =====================================================

ord_features = ['Education']

ord_pipeline = Pipeline([

    ('imputer',
     SimpleImputer(
         strategy='most_frequent'
     )),

    ('ordinal',
     OrdinalEncoder(

         categories=[[
             'Bachelor',
             'Master',
             'PhD'
         ]]
     ))

])

# =====================================================
# NOMINAL FEATURES
# =====================================================

cat_features = [
    'EmploymentType',
    'MaritalStatus',
    'city_cat'
]

cat_pipeline = Pipeline([

    ('imputer',
     SimpleImputer(

         strategy='constant',

         fill_value='Missing'
     )),

    ('ohe',
     OneHotEncoder(

         drop='first',

         handle_unknown='ignore'
     ))

])

# =====================================================
# COLUMN TRANSFORMER
# =====================================================

preprocessor = ColumnTransformer([

    (
        'num',
        num_pipeline,
        num_features
    ),

    (
        'ord',
        ord_pipeline,
        ord_features
    ),

    (
        'cat',
        cat_pipeline,
        cat_features
    )

])

# =====================================================
# FINAL PIPELINE
# =====================================================

pipe = Pipeline([

    ('preprocessing',
     preprocessor),

    ('classifier',
     LogisticRegression())

])

# =====================================================
# GRID SEARCH
# =====================================================

param_grid = {

    'preprocessing__num__knn__n_neighbors':
        [2,3,5],

    'classifier__C':
        [0.1,1,10]
}

grid = GridSearchCV(

    pipe,

    param_grid,

    cv=3,

    scoring='accuracy'
)

# =====================================================
# TRAIN MODEL
# =====================================================

grid.fit(X_train,y_train)

# =====================================================
# BEST PARAMETERS
# =====================================================

print("Best Parameters:")
print(grid.best_params_)

# =====================================================
# PREDICTIONS
# =====================================================

y_pred = grid.predict(X_test)

# =====================================================
# ACCURACY
# =====================================================

print("\nAccuracy:")
print(
    accuracy_score(y_test,y_pred)
)

# =====================================================
# CLASSIFICATION REPORT
# =====================================================

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown ca

Best Parameters:
{'classifier__C': 10, 'preprocessing__num__knn__n_neighbors': 2}

Accuracy:
0.5

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.00      0.00      0.00         0

    accuracy                           0.50         2
   macro avg       0.50      0.25      0.33         2
weighted avg       1.00      0.50      0.67         2



/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_